# 03 Bayesian optimisation and HMM / Viterbi decoding

In [1]:
from pathlib import Path
import json, sys, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve().parents[1]          # report/notebooks -> repository root
sys.path.insert(0, str(ROOT / "src"))
FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
RES = ROOT / "results"
DATA = {"train": ROOT / "student_dataset" / "train", "student_test": ROOT / "student_dataset" / "student_test",
        "evaluator_test": ROOT / "evaluator_test"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold"})
PAL = {"train": "#2b6cb0", "student_test": "#dd6b20", "evaluator_test": "#2f855a"}

def save(name):
    plt.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.show()

def read_whitespace(path, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None, dtype=dtype).to_numpy()

def load_trip(part, tid):
    d = DATA[part] / tid
    return {"track": read_whitespace(d / f"{tid}.track"), "nodes": read_whitespace(d / f"{tid}.nodes"),
            "arcs": read_whitespace(d / f"{tid}.arcs", dtype=np.int64),
            "route": read_whitespace(d / f"{tid}.route", dtype=np.int64).ravel()}

def trips(part):
    return sorted(p.name for p in DATA[part].iterdir() if p.is_dir() and not p.name.startswith("."))

In [2]:
TUNE = RES / "tuning"
git = lambda p: json.load(open(p))
tl, tx, tm = (pd.read_csv(TUNE / f) for f in ["lightgbm_trials.csv", "xgboost_trials.csv", "mlp_trials.csv"])
sl, sx, sm = (git(TUNE / f) for f in ["lightgbm_tuning_summary.json", "xgboost_tuning_summary.json", "mlp_tuning_summary.json"])
gp = pd.read_csv(ROOT / "mlp" / "results" / "bayesian_optimisation_trials.csv"); ml = git(ROOT / "mlp" / "results" / "mlp_complete_report_metrics.json")

def history(ax, df, title):
    done = df[df.state == "COMPLETE"]; pruned = df[df.state == "PRUNED"]
    ax.scatter(done.number, done.value, s=30, color="#2b6cb0", label="completed trial", zorder=3)
    ax.scatter(pruned.number, pruned.value, s=18, color="#a0aec0", marker="x", label="pruned (partial mean)", zorder=2)
    ax.step(done.number, done.value.cummax(), where="post", color="#c05621", label="best so far"); ax.axhline(done.value.iloc[0], ls="--", color="grey", lw=0.8)
    ax.set(title=title, xlabel="trial", ylabel="mean 5-fold CV Top-1"); ax.legend(fontsize=8)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
history(ax[0], tl, "LightGBM (TPE, 30 trials)"); history(ax[1], tx, "XGBoost (TPE, 30 trials)"); history(ax[2], tm, "Deep MLP (TPE, 15 trials)")
plt.tight_layout(); save("tune_01_trial_history.png")
print({k: (len(v), v.state.value_counts().to_dict()) for k, v in {"lightgbm": tl, "xgboost": tx, "mlp": tm}.items()})

{'lightgbm': (30, {'COMPLETE': 19, 'PRUNED': 11}), 'xgboost': (30, {'COMPLETE': 19, 'PRUNED': 11}), 'mlp': (15, {'COMPLETE': 14, 'PRUNED': 1})}


In [3]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].plot(gp.trial, gp.objective, marker="o", color="#2b6cb0", label="objective (0.85 macro F1 + 0.15 Top-1)"); ax[0].plot(gp.trial, gp.cv_top1_accuracy, marker="s", color="#dd6b20", label="CV Top-1")
ax[0].set(xlabel="trial", title="GP-EI search for the Deep MLP (10 trials)"); ax[0].legend(fontsize=8)
c = tl[tl.state == "COMPLETE"]; s = ax[1].scatter(c.params_learning_rate, c.params_num_leaves, c=c.value, cmap="viridis", s=60); ax[1].set(xscale="log", yscale="log", xlabel="learning rate", ylabel="num_leaves", title="LightGBM: completed trials")
plt.colorbar(s, ax=ax[1], label="CV Top-1"); plt.tight_layout(); save("tune_02_gpei_and_lightgbm_space.png")

In [4]:
rows = [
 ("LightGBM", "untuned", sl["untuned_cv_top1_mean"], sl["student_test_untuned"]["top1_acc"], sl["student_test_untuned"]["cand_macro_f1"], sl["student_test_untuned"]["log_loss"], sl["student_test_untuned"]["inference_s"]),
 ("LightGBM", "tuned", sl["best_cv_top1_mean"], sl["student_test_tuned"]["top1_acc"], sl["student_test_tuned"]["cand_macro_f1"], sl["student_test_tuned"]["log_loss"], sl["student_test_tuned"]["inference_s"]),
 ("XGBoost", "untuned", np.nan, sx["student_test_their_untuned"]["top1_acc"], sx["student_test_their_untuned"]["f1_macro"], sx["student_test_their_untuned"]["log_loss"], sx["student_test_their_untuned"]["inference_s"]),
 ("XGBoost", "tuned, search 1", sx["their_tuned_cv_top1_mean"], sx["student_test_their_tuned"]["top1_acc"], sx["student_test_their_tuned"]["f1_macro"], sx["student_test_their_tuned"]["log_loss"], sx["student_test_their_tuned"]["inference_s"]),
 ("XGBoost", "re-tuned, search 2", sx["best_cv_top1_mean"], sx["student_test_retuned"]["top1_acc"], sx["student_test_retuned"]["f1_macro"], sx["student_test_retuned"]["log_loss"], sx["student_test_retuned"]["inference_s"]),
]
tab = pd.DataFrame(rows, columns=["model", "config", "cv_top1_mean", "student_top1", "student_macro_f1", "student_logloss", "inference_s"])
tab.to_csv(TAB / "tuning_comparison_trees.csv", index=False)
mlp_rows = pd.DataFrame([
 ("Deep MLP", "untuned", ml["cv_5fold_mean_macro_f1"], 0.9041, 0.9271, 0.9576, 0.0888, 0.1821),
 ("Deep MLP", "tuned, GP-EI search", 0.9558, 0.9050, 0.9329, 0.9625, 0.0684, 0.2229),
 ("Deep MLP", "re-tuned, TPE search", np.nan, sm["best_cv_top1_mean"], sm["student_test_retuned"]["top1_mean"], np.nan, np.nan, np.nan)],
 columns=["model", "config", "cv_macro_f1", "cv_top1_mean", "student_top1", "student_macro_f1", "student_logloss", "inference_s"])
mlp_rows.to_csv(TAB / "tuning_comparison_mlp.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
lab = [f"{a}\n{b}" for a, b in zip(tab.model, tab.config)]
ax[0].bar(range(len(tab)), tab.cv_top1_mean.fillna(0), color="#2b6cb0"); ax[0].set_ylim(0.8, 0.92); ax[0].set_xticks(range(len(tab))); ax[0].set_xticklabels(lab, fontsize=7); ax[0].set_title("Mean CV Top-1, untuned and tuned (tree models)")
ax[1].bar(range(len(tab)), tab.student_top1, color="#dd6b20"); ax[1].set_ylim(0.8, 0.93); ax[1].set_xticks(range(len(tab))); ax[1].set_xticklabels(lab, fontsize=7); ax[1].set_title("Student-test Top-1, untuned and tuned (tree models)")
for a, col in zip(ax, ["cv_top1_mean", "student_top1"]):
    for i, v in enumerate(tab[col]):
        if not np.isnan(v): a.text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=8)
plt.tight_layout(); save("tune_03_untuned_vs_tuned_trees.png")
mlp_rows.round(4)

,model,config,cv_macro_f1,cv_top1_mean,student_top1,student_macro_f1,student_logloss,inference_s
0,Deep MLP,untuned,0.9553,0.9041,0.9271,0.9576,0.0888,0.1821
1,Deep MLP,"tuned, GP-EI search",0.9558,0.9050,0.9329,0.9625,0.0684,0.2229
2,Deep MLP,"re-tuned, TPE search",NaN,0.9057,0.9319,NaN,NaN,NaN


## HMM / Viterbi against classifier only

In [5]:
ev = pd.read_csv(RES / "evaluator_test" / "evaluator_test_summary.csv"); st = pd.read_csv(RES / "student_test" / "student_test_summary.csv")
MOD = ["LightGBM", "Deep MLP", "XGBoost"]
def pair(df, model):
    return df[(df.model == model) & (df.stage == "classifier only")].iloc[0], df[(df.model == model) & (df.stage.str.contains("HMM"))].iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8)); w = 0.2; x = np.arange(6)
labels = [f"{m}\n{sp}" for m in MOD for sp in ("student", "evaluator")]
a_top, b_top, a_c, b_c = [], [], [], []
for m in MOD:
    for df in (st, ev):
        a, b = pair(df, m); a_top.append(a.top1_acc); b_top.append(b.top1_acc); a_c.append(a.route_continuity); b_c.append(b.route_continuity)
ax[0].bar(x - w / 2, a_top, w, label="classifier only", color="#2b6cb0"); ax[0].bar(x + w / 2, b_top, w, label="classifier + HMM/Viterbi", color="#c05621")
ax[1].bar(x - w / 2, a_c, w, color="#2b6cb0"); ax[1].bar(x + w / 2, b_c, w, color="#c05621")
for a_, vv in ((ax[0], (a_top, b_top)), (ax[1], (a_c, b_c))):
    for i in range(6): a_.text(i - w / 2, vv[0][i] + 0.002, f"{vv[0][i]:.3f}", ha="center", fontsize=7); a_.text(i + w / 2, vv[1][i] + 0.002, f"{vv[1][i]:.3f}", ha="center", fontsize=7)
for a_, t_ in zip(ax, ["Top-1 accuracy", "Route continuity"]): a_.set_xticks(x); a_.set_xticklabels(labels, fontsize=8); a_.set_title(t_)
ax[0].set_ylim(0.75, 1); ax[1].set_ylim(0.85, 1.01); ax[0].legend(fontsize=8); plt.tight_layout(); save("hmm_01_classifier_vs_hmm.png")

In [6]:
pt = pd.read_csv(RES / "evaluator_test" / "evaluator_test_per_trip_top1.csv", index_col=0)
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6)); x = np.arange(len(pt))
for a_, m in zip(ax, MOD):
    a_.bar(x - 0.2, pt[m], 0.4, label="classifier only", color="#2b6cb0"); a_.bar(x + 0.2, pt[m + " + HMM"], 0.4, label="+ HMM/Viterbi", color="#c05621")
    a_.set_xticks(x); a_.set_xticklabels(pt.index, rotation=35, ha="right", fontsize=8); a_.set_ylim(0.5, 1); a_.set_title(f"{m}: per-trip Top-1, evaluator_test"); a_.legend(fontsize=8)
plt.tight_layout(); save("hmm_02_per_trip_gain.png")
pd.DataFrame({m + " gain": pt[m + " + HMM"] - pt[m] for m in MOD}).round(4).T

trip,15,17,23,30,49,59,67,73,87,94
LightGBM gain,0.0608,-0.0653,0.0349,0.0153,-0.0030,-0.0621,-0.0054,0.0009,0.0215,0.0003
Deep MLP gain,0.0054,-0.0052,0.0000,0.0030,0.0019,0.0020,0.0011,0.0008,0.0079,0.0058
XGBoost gain,0.2956,-0.0185,0.0283,0.0367,0.0095,-0.0252,0.0524,0.0245,0.0522,0.0975


In [7]:
ht = pd.read_csv(ROOT / "xg boost" / "results" / "hmm_trials.csv")
print(list(ht.columns))
col = [c for c in ht.columns if any(k in c.lower() for k in ("top1", "value", "objective", "acc"))][0]
fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(ht.index, ht[col], marker="o", color="#2b6cb0", label=col); ax.plot(ht.index, ht[col].cummax(), color="#c05621", label="best so far")
ax.set(xlabel="trial", ylabel=col, title="HMM parameter search (40 trials)"); ax.legend(); plt.tight_layout(); save("hmm_03_xgb_hmm_search.png")

['number', 'value', 'datetime_start', 'datetime_complete', 'duration', 'params_alpha', 'params_disc', 'params_gap_s', 'params_lam', 'params_uturn', 'system_attrs_tpe:relative_params:0', 'state']
